# MVTec AD Data Exploration and Preprocessing

This notebook performs only reproducible MVTec AD inspection, validation, preprocessing, and reporting. Raw data is treated as read-only.

## 1. Environment Setup and Dataset Inspection

Configure paths once, inspect the actual directory structure, and retain a raw-file checksum manifest so later cells can prove the raw dataset was not changed.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import numpy as np
from PIL import Image, UnidentifiedImageError

RAW_DATA_DIR = Path('datasets/mvtec_ad')
PROCESSED_DATA_DIR = Path('datasets/mvtec_ad_processed')
IMAGE_SIZE = 256
SUPPORTED_IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
RGB_RESAMPLE = Image.Resampling.LANCZOS
MASK_RESAMPLE = Image.Resampling.NEAREST

assert RAW_DATA_DIR.is_dir(), f'Raw dataset directory not found: {RAW_DATA_DIR.resolve()}'
assert RAW_DATA_DIR.resolve() != PROCESSED_DATA_DIR.resolve(), 'Raw and processed paths must differ.'

def is_supported_image(path):
    return path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS

def image_files(directory):
    return sorted(path for path in directory.rglob('*') if is_supported_image(path))

def category_directories(root=RAW_DATA_DIR):
    required = {'train', 'test', 'ground_truth'}
    return sorted(path for path in root.iterdir() if path.is_dir() and required.issubset({item.name for item in path.iterdir() if item.is_dir()}))

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def dataset_manifest(root):
    return {str(path.relative_to(root)): (path.stat().st_size, file_sha256(path)) for path in sorted(root.rglob('*')) if path.is_file()}

CATEGORIES = category_directories()
assert CATEGORIES, f'No valid MVTec categories found under {RAW_DATA_DIR}'
RAW_FILE_MANIFEST = dataset_manifest(RAW_DATA_DIR)

print('MVTec AD Dataset')
print('=' * 50)
print(f'Raw dataset: {RAW_DATA_DIR.resolve()}')
print(f'Categories discovered: {len(CATEGORIES)}')
print('Categories:', ', '.join(category.name for category in CATEGORIES))
for category in CATEGORIES:
    print(f'\nCategory: {category.name}')
    for split in ('train', 'test', 'ground_truth'):
        split_dir = category / split
        class_counts = {subdir.name: len(image_files(subdir)) for subdir in sorted(split_dir.iterdir()) if subdir.is_dir()}
        print(f'  {split}/')
        for class_name, count in class_counts.items():
            print(f'    {class_name}: {count} image files')

root_non_directories = [path.name for path in RAW_DATA_DIR.iterdir() if not path.is_dir()]
print(f'\nRaw file manifest entries: {len(RAW_FILE_MANIFEST)}')
print('Unexpected top-level non-directory entries:', root_non_directories or 'none')